# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de Google Gemini

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [ ]:
from google import genai

# Reemplazá esto por tu API key de Google AI Studio (https://aistudio.google.com/apikey)
API_KEY = "AQ"

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explicá en una sola frase qué es la búsqueda semántica (retrieval)."
)

print(response.text)

La búsqueda semántica (retrieval) recupera información al comprender el *significado* y la *intención* de una consulta, más allá de la coincidencia literal de palabras clave, para ofrecer resultados contextualmente más relevantes.


## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [2]:
from sklearn.datasets import fetch_20newsgroups
import random

# Usamos algunas categorías para tener un subset chico y variado
categorias = ['sci.space', 'rec.sport.hockey', 'comp.graphics', 'talk.politics.mideast']

newsgroups = fetch_20newsgroups(
    subset='train',
    categories=categorias,
    remove=('headers', 'footers', 'quotes')
)

random.seed(42)
n_docs = 250
indices = random.sample(range(len(newsgroups.data)), n_docs)

documentos = [newsgroups.data[i].strip() for i in indices if newsgroups.data[i].strip()]
etiquetas = [newsgroups.target_names[newsgroups.target[i]] for i in indices if newsgroups.data[i].strip()]

print(f"Cantidad de documentos cargados: {len(documentos)}")
print("\nEjemplo de documento:\n")
print(documentos[0][:500])

Cantidad de documentos cargados: 241

Ejemplo de documento:

----- News saved at 23 Apr 93 22:22:40 GMT
  



Well, I'm working on it, but getting a little impatient. So far, 
I've made it through Egyptian, Chinese, and Greek cultures, and
up through the Rennaisance. But so far, these insights just don't 
seem to be gelling. Perhaps it's in an appendix somewhere.


### 2.2 Transformo a embeddings

In [5]:
import time
import numpy as np

def obtener_embeddings(textos, task_type="RETRIEVAL_DOCUMENT", batch_size=5):
    """Llama a la API de embeddings con delay más largo entre lotes"""
    embeddings = []
    for i in range(0, len(textos), batch_size):
        lote = [t[:2000] for t in textos[i:i + batch_size]]
        respuesta = client.models.embed_content(
            model="gemini-embedding-001",
            contents=lote,
            config=types.EmbedContentConfig(task_type=task_type)
        )
        embeddings.extend([item.values for item in respuesta.embeddings])
        print(f"Procesados {min(i + batch_size, len(textos))}/{len(textos)} documentos")
        time.sleep(3)  # ⬅️ Aumenta de 1s a 3s (o más si es necesario)
    return np.array(embeddings)

embeddings_docs = obtener_embeddings(documentos, task_type="RETRIEVAL_DOCUMENT")

Procesados 5/241 documentos
Procesados 10/241 documentos
Procesados 15/241 documentos
Procesados 20/241 documentos
Procesados 25/241 documentos
Procesados 30/241 documentos
Procesados 35/241 documentos
Procesados 40/241 documentos
Procesados 45/241 documentos
Procesados 50/241 documentos
Procesados 55/241 documentos
Procesados 60/241 documentos
Procesados 65/241 documentos
Procesados 70/241 documentos
Procesados 75/241 documentos
Procesados 80/241 documentos
Procesados 85/241 documentos
Procesados 90/241 documentos
Procesados 95/241 documentos
Procesados 100/241 documentos
Procesados 105/241 documentos
Procesados 110/241 documentos
Procesados 115/241 documentos
Procesados 120/241 documentos
Procesados 125/241 documentos
Procesados 130/241 documentos
Procesados 135/241 documentos
Procesados 140/241 documentos
Procesados 145/241 documentos
Procesados 150/241 documentos
Procesados 155/241 documentos
Procesados 160/241 documentos
Procesados 165/241 documentos
Procesados 170/241 documentos


### 2.3 Creo una query y hago la búsqueda

In [6]:
query = "¿Cómo funciona la exploración espacial y los viajes a la Luna?"

embedding_query = obtener_embeddings([query], task_type="RETRIEVAL_QUERY")[0]

print(f"Forma del embedding de la query: {embedding_query.shape}")

Procesados 1/1 documentos
Forma del embedding de la query: (3072,)


Obtengo los 5 documentos más similares a mi query

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

similitudes = cosine_similarity([embedding_query], embeddings_docs)[0]

top_5_indices = np.argsort(similitudes)[::-1][:5]

print("Top 5 documentos más similares a la query:\n")
for rank, idx in enumerate(top_5_indices, start=1):
    print(f"--- Puesto {rank} | Categoría: {etiquetas[idx]} | Similitud: {similitudes[idx]:.4f} ---")
    print(documentos[idx][:300])
    print()

Top 5 documentos más similares a la query:

--- Puesto 1 | Categoría: sci.space | Similitud: 0.6638 ---
Archive-name: space/intro
Last-modified: $Date: 93/04/01 14:39:10 $

    FREQUENTLY ASKED QUESTIONS ON SCI.SPACE/SCI.ASTRO

    INTRODUCTION

    This series of linked messages is periodically posted to the Usenet
groups sci.space and sci.astro in an attempt to provide good answers to
frequently ask

--- Puesto 2 | Categoría: sci.space | Similitud: 0.6585 ---
Hey!  My dad has an old hangar and Judy has some old rockets in her attic,
let's put on a Lunar program! . . .  Sounds good, but . . .
Let's play a game - What would be a reasonable reward?  What companies would
have a reasonable shot at pulling off such a feat?  Just where in the
budget would the r

--- Puesto 3 | Categoría: sci.space | Similitud: 0.6528 ---
: Announce that a reward of $1 billion would go to the first corporation 
: who successfully keeps at least 1 person alive on the moon for a year. 
: Then you'd see some of

### Pasar los documentos recuperados a Gemini para que responda la query

In [8]:
contexto = "\n\n---\n\n".join([documentos[idx] for idx in top_5_indices])

prompt_rag = f"""Respondé la siguiente pregunta usando únicamente la información de los documentos de contexto.
Si la respuesta no está en el contexto, decilo explícitamente.

Contexto:
{contexto}

Pregunta: {query}

Respuesta:"""

respuesta_final = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_rag
)

print(respuesta_final.text)

La información proporcionada en el contexto es una introducción a una serie de mensajes de preguntas frecuentes (FAQ) sobre sci.space/sci.astro y no describe cómo funciona la exploración espacial o los viajes a la Luna.

El documento menciona que se abordan temas relacionados en otras publicaciones de la serie FAQ, como:
*   Cálculos y formatos de datos (publicación #4)
*   Referencias sobre áreas específicas, incluyendo propulsión de cohetes y diseño de naves espaciales (publicación #5)
*   Respuestas sobre el transbordador espacial y su funcionamiento, como por qué el transbordador gira justo después del despegue y la composición del combustible de los cohetes de propulsión sólida (publicación #9)
*   Misiones planetarias históricas y futuras (publicaciones #10 y #11)

Además, el contexto incluye discusiones sobre modelos de financiación para la actividad lunar (recompensas, leyes de compra de datos) y la rentabilidad de una base lunar, pero no explica los mecanismos o el funcionamie